# Data Preprocessing (Structural)

Fetch structural brain measures (FreeSurfer) from ABIDE, clean them, and save them ready for training.


**About ABIDE** 

The Autism Brain Imaging Data Exchange (ABIDE) dataset provides previously collected resting state functional magnetic resonance imaging datasets from 539 individuals with ASD and 573 typical controls for the purpose of data sharing in the broader scientific community. 

Besides the fMRI data, the ABIDE Preprocessed project provides the FreeSurfer (v5.1) outputs of the structural MRI scans, such as the volume of each subcortical region and the total gray matter volume.

In [1]:
# Import
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset
import os
import json
import urllib.request

## Phenotypic Data

The phenotypic data metadata was retrieved from http://fcon_1000.projects.nitrc.org/indi/abide/abide_I.html, in the "ABIDE I Downloads" section, specifically the *Phenotypic Data Legend* download

Selecting the healthy controls (`DX_GROUP == 2`) that passed the anatomical quality control, and extracting `FILE_ID`, `AGE_AT_SCAN` and `SEX`.

In [2]:
# ABIDE base url
abide_url = "https://fcp-indi.s3.amazonaws.com/data/Projects/ABIDE_Initiative"

# Phenotypic data (download only if not already in the data folder)
pheno_path = "../data/ABIDE_pcp/Phenotypic_V1_0b_preprocessed1.csv"
if not os.path.exists(pheno_path):
    os.makedirs(os.path.dirname(pheno_path), exist_ok=True)
    urllib.request.urlretrieve(f"{abide_url}/Phenotypic_V1_0b_preprocessed1.csv", pheno_path)

df = pd.read_csv(pheno_path)
print(f"Total subjects: {len(df)}")
print(df.columns)
df.head(3)

Total subjects: 1112
Index(['Unnamed: 0.1', 'Unnamed: 0', 'SUB_ID', 'X', 'subject', 'SITE_ID',
       'FILE_ID', 'DX_GROUP', 'DSM_IV_TR', 'AGE_AT_SCAN',
       ...
       'qc_notes_rater_1', 'qc_anat_rater_2', 'qc_anat_notes_rater_2',
       'qc_func_rater_2', 'qc_func_notes_rater_2', 'qc_anat_rater_3',
       'qc_anat_notes_rater_3', 'qc_func_rater_3', 'qc_func_notes_rater_3',
       'SUB_IN_SMP'],
      dtype='object', length=106)


,Unnamed: 0.1,Unnamed: 0,SUB_ID,X,subject,SITE_ID,FILE_ID,DX_GROUP,DSM_IV_TR,AGE_AT_SCAN,...,qc_notes_rater_1,qc_anat_rater_2,qc_anat_notes_rater_2,qc_func_rater_2,qc_func_notes_rater_2,qc_anat_rater_3,qc_anat_notes_rater_3,qc_func_rater_3,qc_func_notes_rater_3,SUB_IN_SMP
0,0,1,50002,1,50002,PITT,no_filename,1,1,16.77,...,NaN,OK,NaN,fail,ic-parietal-cerebellum,OK,NaN,fail,ERROR #24,1
1,1,2,50003,2,50003,PITT,Pitt_0050003,1,1,24.45,...,NaN,OK,NaN,OK,NaN,OK,NaN,OK,NaN,1
2,2,3,50004,3,50004,PITT,Pitt_0050004,1,1,19.09,...,NaN,OK,NaN,OK,NaN,OK,NaN,OK,NaN,1


In [3]:
# Filter for Healthy Controls (DX_GROUP == 2) with available data
df = df[(df['DX_GROUP'] == 2) & (df['FILE_ID'] != 'no_filename')].copy()

# Anatomical quality control (same raters as Nilearn, anatomical only)
df = df[(df['qc_rater_1'] == 'OK') &
        (df['qc_anat_rater_2'].isin(['OK', 'maybe'])) &
        (df['qc_anat_rater_3'] == 'OK')].copy()

# Select ID, Site, Age and Sex
df = df[['FILE_ID', 'SITE_ID', 'AGE_AT_SCAN', 'SEX']].copy()

# Map SEX (ABIDE uses 1=Male, 2=Female) to 0 and 1
df['SEX'] = df['SEX'].map({1: 0, 2: 1})

print(f"Healthy controls: {len(df)}")
df.head()

Healthy controls: 477


,FILE_ID,SITE_ID,AGE_AT_SCAN,SEX
26,Pitt_0050030,PITT,25.12,0
27,Pitt_0050031,PITT,12.92,0
28,Pitt_0050032,PITT,19.80,0
29,Pitt_0050033,PITT,12.15,0
30,Pitt_0050034,PITT,14.77,0


## FreeSurfer Data

For each subject, the file `aseg.stats` contains the volume (mm³) of each subcortical region (e.g. hippocampus), and whole brain measures (e.g. total gray matter volume, intracranial volume).

Files are downloaded from `{abide_url}/Outputs/freesurfer/5.1/<FILE_ID>/stats/aseg.stats` into `../data/freesurfer`.

In [4]:
# Download aseg.stats for each subject (skip the ones already downloaded)
fs_dir = "../data/freesurfer"
os.makedirs(fs_dir, exist_ok=True)

missing = []
for file_id in df['FILE_ID']:
    local_path = f"{fs_dir}/{file_id}_aseg.stats"
    if os.path.exists(local_path):
        continue
    try:
        urllib.request.urlretrieve(f"{abide_url}/Outputs/freesurfer/5.1/{file_id}/stats/aseg.stats", local_path)
    except Exception:
        missing.append(file_id)

print(f"Files in {fs_dir}: {len(os.listdir(fs_dir))}")
print(f"Subjects without aseg.stats: {len(missing)}")

Files in ../data/freesurfer: 477
Subjects without aseg.stats: 0
